# NBA Team Performance Analysis

## Overview

This project uses 30 years of NBA team statistics (1996–2026) to answer one core question:

**Do teams win as many games as their stats suggest they should?**

Using Ridge Regression, I predict how many games each team *should* win based on 17 statistical inputs — both offensive metrics (shooting percentages, assists, rebounds, steals, turnovers, and blocks) and opponent/defensive metrics (opponent shooting percentages, opponent points, opponent assists, and opponent rebounds). The gap between what a team actually won and what the model predicted is called the **residual**. A consistently positive residual signals something beyond raw talent: great coaching, culture, or clutch performance. A consistently negative one suggests a team that underperformed its potential.

---

## Table of Contents

- [Why Ridge Regression?](#Why-Ridge-Regression?)
- [Data Collection and Model](#Data-Collection-and-Model)
- [Understanding the Model Metrics](#Understanding-the-Model-Metrics)
- [Team Performance Charts](#Team-Performance-Charts)
- [How to Draw Conclusions from These Charts](#How-to-Draw-Conclusions-from-These-Charts)
- [Coach Tenure Data](#Coach-Tenure-Data)
- [All-Seasons Model and Coach Residuals](#All-Seasons-Model-and-Coach-Residuals)
- [League-Wide Coaching Rankings](#League-Wide-Coaching-Rankings)
- [Team Coaching History](#Team-Coaching-History)
- [Coach Career Breakdown](#Coach-Career-Breakdown)
- [Most Surprising Seasons — Era Adjusted](#Most-Surprising-Seasons)

---

## What This Project Produces

1. **Pulls data** from the official NBA API — every team, every season from 1996–97 through 2025–26
2. **Trains a walk-forward model** that predicts wins using only stats from *past* seasons — no future information ever leaks in
3. **Calculates residuals** — the gap between actual wins and model predictions
4. **Shows 30-year franchise charts** with an interactive dropdown for any NBA team
5. **Ranks every NBA coach** by how much they consistently beat or missed the model's expectations
6. **Provides three interactive coaching views** — league-wide rankings, team-by-team history, and season-by-season career breakdowns

---

## Tools Used

- `nba_api` — fetches data directly from NBA.com
- `pandas` — stores and manipulates the dataset
- `scikit-learn` — builds the Ridge Regression model with automatic alpha tuning via `RidgeCV`
- `matplotlib` — generates all charts
- `ipywidgets` — powers the interactive dropdowns

## Why Ridge Regression?

There are many models that could predict wins from stats — so why Ridge Regression specifically?

NBA team statistics are highly correlated with each other. Teams that shoot well tend to assist well. Teams that rebound well tend to defend well. This is even more pronounced once opponent stats are added — a team that defends well tends to both limit opponent shooting percentages *and* reduce opponent assists and rebounds simultaneously. When input features are this correlated, standard linear regression can go haywire — it assigns huge positive coefficients to some features and compensating negative coefficients to others in ways that don't make intuitive sense and don't generalize to new seasons.

**Ridge Regression** solves this by adding a penalty for large coefficients. It forces the model to spread the weight more evenly across all 17 inputs rather than over-relying on any one. The result is a more stable, more interpretable model that generalises honestly across 30 years of NBA data.

The strength of this penalty is controlled by the **alpha parameter**. Rather than picking alpha manually, this project uses **RidgeCV**, which automatically tests eleven candidate values (1e-4, 1e-3, 0.01, 0.05, 0.1, 0.5, 1.0, 5.0, 10.0, 50.0, 100.0) and selects the best one via cross-validation on each training window. The chosen alpha is printed at runtime so you can see exactly what the model settled on. With the full 17-feature set, RidgeCV consistently selects alphas in the 0.5–10.0 range, applying stronger smoothing than the 9-feature baseline needed.

In short: Ridge Regression was chosen because it is simple enough to interpret, robust enough to handle correlated stats, and honest enough to generalise across different eras of basketball without overfitting.

In [5]:
# list of every season data is pulled from
seasons = [
    '1996-97', '1997-98', '1998-99', '1999-00',
    '2000-01', '2001-02', '2002-03', '2003-04',
    '2004-05', '2005-06', '2006-07', '2007-08',
    '2008-09', '2009-10', '2010-11', '2011-12',
    '2012-13', '2013-14', '2014-15', '2015-16',
    '2016-17', '2017-18', '2018-19', '2019-20',
    '2020-21', '2021-22', '2022-23', '2023-24',
    '2024-25', '2025-26'
]

In [55]:
from nba_api.stats.endpoints import leaguedashteamstats
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import r2_score, mean_squared_error
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pandas as pd
import numpy as np
import time
import ipywidgets as widgets
from IPython.display import display, Image
import io
%matplotlib inline

# ── 1. Fetch base team stats ──────────────────────────────────────────────────
base_dfs = []
for s in seasons:
    try:
        t = leaguedashteamstats.LeagueDashTeamStats(season=s, measure_type_detailed_defense='Base').get_data_frames()[0]
        t['SEASON'] = s
        base_dfs.append(t)
        print(f"✓ base {s}")
        time.sleep(0.6)
    except Exception as e:
        print(f"✗ base {s}: {e}")
df_base = pd.concat(base_dfs, ignore_index=True)

# ── 2. Fetch opponent stats ───────────────────────────────────────────────────
opp_dfs = []
for s in seasons:
    try:
        t = leaguedashteamstats.LeagueDashTeamStats(season=s, measure_type_detailed_defense='Opponent').get_data_frames()[0]
        t['SEASON'] = s
        opp_dfs.append(t)
        print(f"✓ opp  {s}")
        time.sleep(0.6)
    except Exception as e:
        print(f"✗ opp  {s}: {e}")
df_opp = pd.concat(opp_dfs, ignore_index=True)

# ── 3. Merge base + opponent ──────────────────────────────────────────────────
opp_keep = ['TEAM_ID','SEASON'] + [c for c in df_opp.columns if c.startswith('OPP_')]
df_new = df_base.merge(df_opp[opp_keep], on=['TEAM_ID','SEASON'], how='left')

opp_cols = [c for c in df_new.columns if c.startswith('OPP_')]
print(f"\nOpponent columns merged: {len(opp_cols)}")
print(f"Non-null OPP_FG_PCT rows: {df_new['OPP_FG_PCT'].notna().sum()} / {len(df_new)}")

# ── 4. Feature engineering ────────────────────────────────────────────────────
df_new['W_PCT'] = df_new['W'] / df_new['GP']
scaler = StandardScaler()
df_new['W_PCT_scaled'] = scaler.fit_transform(df_new[['W_PCT']])

for c in ['AST','OREB','DREB','STL','TOV','BLK']:
    df_new[f'{c}_PG'] = df_new[c] / df_new['GP']
for c in ['OPP_PTS','OPP_AST','OPP_OREB','OPP_DREB','OPP_STL','OPP_TOV','OPP_BLK']:
    if c in df_new.columns:
        df_new[f'{c}_PG'] = df_new[c] / df_new['GP']

base_feats = ['FG_PCT','FG3_PCT','FT_PCT','AST_PG','OREB_PG','DREB_PG','STL_PG','TOV_PG','BLK_PG',
              'OPP_FG_PCT','OPP_FG3_PCT','OPP_FT_PCT',
              'OPP_PTS_PG','OPP_AST_PG','OPP_OREB_PG','OPP_DREB_PG','OPP_TOV_PG']
feats = [f for f in base_feats if f in df_new.columns]
print(f"\nFeatures used ({len(feats)}): {feats}")

# ── 5. Walk-forward validation + metrics ──────────────────────────────────────
alphas = [1e-4, 1e-3, 1e-2, 0.05, 0.1, 0.5, 1.0, 5.0, 10.0, 50.0, 100.0]
metrics, pred_rows = [], []

for i, season in enumerate(seasons[1:], 1):
    if season < '1999-00':
        continue
    train = df_new[df_new['SEASON'].isin(seasons[:i])]
    test  = df_new[df_new['SEASON'] == season]
    X_tr = train[feats].apply(pd.to_numeric, errors='coerce').dropna()
    y_tr = train.loc[X_tr.index, 'W_PCT_scaled']
    X_te = test[feats].apply(pd.to_numeric, errors='coerce').dropna()
    y_te = test.loc[X_te.index, 'W_PCT_scaled']
    if len(X_tr) < 10 or len(X_te) == 0:
        continue
    pipe = make_pipeline(StandardScaler(), RidgeCV(alphas=alphas, cv=5)).fit(X_tr, y_tr)
    y_pred = pipe.predict(X_te)
    # metrics
    metrics.append({
        'season': season,
        'r2':     round(r2_score(y_te, y_pred), 3),
        'rmse':   round(np.sqrt(mean_squared_error(y_te, y_pred)), 3),
        'alpha':  round(pipe.named_steps['ridgecv'].alpha_, 4),
        'n_train': len(X_tr), 'n_test': len(X_te)
    })
    # store predictions for df
    for idx, pred in zip(X_te.index, y_pred):
        pred_rows.append({'idx': idx, 'predicted': pred})

mdf = pd.DataFrame(metrics)
print("\n── Model Performance")
print(f"  Avg R²  : {mdf['r2'].mean():.3f}")
print(f"  Avg RMSE: {mdf['rmse'].mean():.3f}")
print(f"  Alpha range: {mdf['alpha'].min()} – {mdf['alpha'].max()}")
print(mdf.to_string(index=False))

# ── 6. Build df (walk-forward residuals) ─────────────────────────────────────
pred_df = pd.DataFrame(pred_rows).set_index('idx')
df_new['predicted'] = pred_df['predicted']
df_new['residual']  = df_new['W_PCT_scaled'] - df_new['predicted']
df_new['residual_vs_era'] = df_new.groupby('SEASON')['residual'].transform(lambda x: x - x.mean())
df = df_new.dropna(subset=['residual']).copy()
print(f"\ndf ready: {len(df)} rows")

# ── 7. Build df_full (all-seasons residuals) ──────────────────────────────────
X_all = df_new[feats].apply(pd.to_numeric, errors='coerce').dropna()
y_all = df_new.loc[X_all.index, 'W_PCT_scaled']
pipe_full = make_pipeline(StandardScaler(), RidgeCV(alphas=alphas, cv=5)).fit(X_all, y_all)
df_full = df_new.copy()
df_full['predicted_full'] = np.nan
df_full.loc[X_all.index, 'predicted_full'] = pipe_full.predict(X_all)
df_full['residual']       = df_full['W_PCT_scaled'] - df_full['predicted_full']
df_full['residual_vs_era'] = df_full.groupby('SEASON')['residual'].transform(lambda x: x - x.mean())
df_full = df_full.dropna(subset=['residual']).copy()
print(f"df_full ready: {len(df_full)} rows")

✓ base 1996-97
✓ base 1997-98
✓ base 1998-99
✓ base 1999-00
✓ base 2000-01
✓ base 2001-02
✓ base 2002-03
✓ base 2003-04
✓ base 2004-05
✓ base 2005-06
✓ base 2006-07
✓ base 2007-08
✓ base 2008-09
✓ base 2009-10
✓ base 2010-11
✓ base 2011-12
✓ base 2012-13
✓ base 2013-14
✓ base 2014-15
✓ base 2015-16
✓ base 2016-17
✓ base 2017-18
✓ base 2018-19
✓ base 2019-20
✓ base 2020-21
✓ base 2021-22
✓ base 2022-23
✓ base 2023-24
✓ base 2024-25
✓ base 2025-26
✓ opp  1996-97
✓ opp  1997-98
✓ opp  1998-99
✓ opp  1999-00
✓ opp  2000-01
✓ opp  2001-02
✓ opp  2002-03
✓ opp  2003-04
✓ opp  2004-05
✓ opp  2005-06
✓ opp  2006-07
✓ opp  2007-08
✓ opp  2008-09
✓ opp  2009-10
✓ opp  2010-11
✓ opp  2011-12
✓ opp  2012-13
✓ opp  2013-14
✓ opp  2014-15
✓ opp  2015-16
✓ opp  2016-17
✓ opp  2017-18
✓ opp  2018-19
✓ opp  2019-20
✓ opp  2020-21
✓ opp  2021-22
✓ opp  2022-23
✓ opp  2023-24
✓ opp  2024-25
✓ opp  2025-26

Opponent columns merged: 40
Non-null OPP_FG_PCT rows: 892 / 892

Features used (17): ['FG_PCT', 'FG

## Data Collection and Model

The cells above do five things in sequence:

**1. Pulls data** from the NBA API for every team across all 30 seasons (1996–97 through 2025–26). Each season is fetched separately with a short delay to avoid rate limiting. A ✓ or ✗ prints for each season as it loads. The improved model fetches two datasets per season: base team stats and opponent stats, then merges them.

**2. Prepares the features.** Win percentage is used as the target (rather than raw wins) so that lockout and shortened seasons — 1998-99 (50 games), 2011-12 (66 games), 2019-20 and 2020-21 (72 games each) — are comparable to full 82-game seasons. Counting stats are converted to per-game rates for the same reason. The 17 features the model trains on are:

| Feature | What it measures |
|---|---|
| `FG_PCT` | Overall field goal shooting efficiency |
| `FG3_PCT` | Three-point shooting efficiency |
| `FT_PCT` | Free throw shooting efficiency |
| `AST_PG` | Assists per game (ball movement / unselfishness) |
| `OREB_PG` | Offensive rebounds per game (second-chance points) |
| `DREB_PG` | Defensive rebounds per game (stopping opponent second chances) |
| `STL_PG` | Steals per game (ball-hawk defence) |
| `TOV_PG` | Turnovers per game (lower = better) |
| `BLK_PG` | Blocks per game (rim protection) |
| `OPP_FG_PCT` | Opponent field goal percentage (defensive quality) |
| `OPP_FG3_PCT` | Opponent three-point percentage (perimeter defence) |
| `OPP_FT_PCT` | Opponent free throw percentage |
| `OPP_PTS_PG` | Opponent points per game conceded |
| `OPP_AST_PG` | Opponent assists per game conceded |
| `OPP_OREB_PG` | Opponent offensive rebounds per game conceded |
| `OPP_DREB_PG` | Opponent defensive rebounds per game |
| `OPP_TOV_PG` | Opponent turnovers per game forced |

**3. Trains a walk-forward model.** This is the most important design choice. The model never uses future information — to predict the 1999-00 season it trains only on 1996–97 through 1998–99; to predict 2000-01 it trains on four seasons; and so on, growing by one season each year. This mirrors how a real analyst would actually use the model. By 2025-26, it has learned from nearly 30 full seasons.

**4. Calculates residuals.** For each team-season, the residual is: *actual scaled win % minus predicted scaled win %*. Positive = overperformed; negative = underperformed. Residuals are also normalised within each season (`residual_vs_era`) so that a team from the early 2000s — when the model had less training data — can be fairly compared to a team from 2020.

**5. Reports model accuracy.** R², RMSE, and alpha are printed for every season. The improved 17-feature model achieves an average R² of 0.862 and average RMSE of 0.355 — a major improvement over the 9-feature baseline (avg R²: 0.443). Adding opponent/defensive stats more than doubled explanatory power.

## Understanding the Model Metrics

After the model runs, it prints a summary table with columns for every season. Here is what each one means:

| Column | What it means |
|---|---|
| **R²** (R-squared) | How much of the variation in team wins the model successfully explains, on a scale of 0 to 1. An R² of 0.86 means 86% of the differences in win percentage across teams is captured by the 17 stats. The improved model (with defensive/opponent features) achieves an average R² of 0.862 — a strong fit for 30 years of NBA data. |
| **RMSE** (Root Mean Squared Error) | The typical size of the model's prediction error, measured in the same units as the target (scaled win %). This model averages 0.355 RMSE.|
| **Alpha chosen** | The regularization strength RidgeCV selected for that season. The improved model uses alphas in the range 0.5–10.0 (selected automatically from a wider candidate grid), applying stronger smoothing than the original baseline model. |
| **n_train** | The number of team-seasons used to train the model for that prediction year. The model grows by roughly 29–30 rows each season (one row per team). Higher n_train generally means more stable and accurate predictions. |

In plain English: R² tells you how well the model fits; RMSE tells you how far off individual predictions tend to be; alpha tells you how much regularization was applied; and n_train tells you how much historical data the model had available.

**Why does the alpha vary?**

**Growing training size** The model starts with just ~58 team-seasons (predicting 1999-00) and grows by ~30 rows each year, reaching ~833 by 2025-26. With more data, the model has more signal to learn from, so the optimal amount of regularization can shift.

**Changing feature correlations over time** As the NBA evolves (e.g., the rise of the three-point era, pace changes), the correlations between features like FG3_PCT, OPP_FG3_PCT, and AST_PG change. When features are more multicollinear, RidgeCV tends to select a higher alpha to counteract that instability. This is why you see 5.0 appearing repeatedly in the mid-2000s to 2010s range, and 0.5–1.0 dominating in other eras.

**Variance in specific training windows** Some seasons had unusual league-wide patterns (e.g., the 2004-05 lockout-shortened season or the 2019-20 bubble season), which affects the cross-validated error surface and nudges the optimal alpha up or down.

In [59]:
# ── Interactive Team Chart ────────────────────────────────────────────────────
ALL_TEAMS = sorted(df['TEAM_NAME'].unique().tolist())

team_dropdown = widgets.Dropdown(
    options=ALL_TEAMS,
    value='Houston Rockets',
    description='Team:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px')
)

team_output = widgets.Output()

def plot_team(team_name):
    mask = df['TEAM_NAME'] == team_name
    team_data = df.loc[mask].sort_values('SEASON')

    if team_data.empty:
        print(f"No data found for {team_name}")
        return

    fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

    axes[0].plot(team_data['SEASON'], team_data['residual'], marker='o', color='steelblue', label=team_name)
    axes[0].axhline(y=0, color='red', linestyle='--', linewidth=1.2, label='Model prediction (zero = met expectations)')
    axes[0].set_ylabel('Residual (Scaled Win %)')
    axes[0].set_title(f'{team_name} — Over/Underperformance vs Model Prediction')
    axes[0].legend(fontsize=8)

    for _, row in team_data.iterrows():
        axes[0].annotate(f"{row['residual']:.2f}", (row['SEASON'], row['residual']),
                        textcoords='offset points', xytext=(0, 6), fontsize=7, ha='center')

    axes[1].plot(team_data['SEASON'], team_data['W'], marker='o', color='darkorange', label=team_name)
    axes[1].set_ylabel('Actual Wins')
    axes[1].set_title('Actual Wins Per Season')

    for _, row in team_data.iterrows():
        axes[1].annotate(f"{int(row['W'])}", (row['SEASON'], row['W']),
                        textcoords='offset points', xytext=(0, 6), fontsize=7, ha='center')

    for ax in axes:
        ax.legend(loc='upper right', fontsize=7)
        ax.set_xlabel('Season')
        ax.tick_params(labelbottom=True, axis='x', labelrotation=90, labelsize=9)

    plt.tight_layout()
    buf = io.BytesIO()
    plt.savefig(buf, format='png', bbox_inches='tight')
    buf.seek(0)
    plt.close(fig)
    display(Image(data=buf.read()))

team_dropdown.observe(on_team_change, names='value')

display(widgets.VBox([
    widgets.Label('Select a team to view its 30-year performance:'),
    team_dropdown,
    team_output
]))

# Trigger initial plot
with team_output:
    plot_team(team_dropdown.value)

## Team Performance Charts

The interactive chart above shows the full 30-year story of any NBA franchise. Use the dropdown to select a team.

**Top chart — Residual over time.** Each point is one season. Points *above* the red dashed line (zero) mean the team won more games than the model predicted from their stats alone. Points *below* it mean they left wins on the table.

**Bottom chart — Actual wins over time.** The raw win total each season, so you can see the team's absolute quality alongside the residual. A team can have a losing record but a *positive* residual (squeezed everything from a bad roster) or a great record but a *negative* residual (underachieved despite elite stats).

Reading both charts together tells the complete story — when wins and residuals align, the performance was fully explained by talent. When they diverge, something else is going on.

**Note about residuals.** A residual around 0 doesn't neccesarily mean the team underperformed, it just means the team performed as expected. For example the 2016-17 Golden State Warriors have a negative residual, but they won 67 games. This is because the model already predicted them to win so many games that when they slightly won less than that number, their residual is below 0. Not because they were a bad team but because they were already expected to win.

---

## How to Draw Conclusions from These Charts

Each section of this project answers a different question. Here is a guide to reading them together.

### Team Residual Chart (top chart above)
- **Sustained positive residuals** over multiple seasons → the team is consistently winning more than their stats justify. This usually signals excellent coaching, a strong culture, or elite clutch performance.
- **Sustained negative residuals** → the team is underperforming its statistical talent. Look for coaching instability, chemistry issues, or roster misuse.
- **A sudden spike or drop** → something changed that year — a new coach, a key trade, or a star injury. Cross-reference with the actual wins chart below to confirm.

### Actual Wins Chart (bottom chart above)
- Use this to distinguish between *talent-driven* and *execution-driven* success. A team with high wins *and* a positive residual has elite stats *and* executes beyond them. A team with high wins *but* a negative residual is winning on raw talent alone — underperforming relative to their potential.
- A team with low wins but a *positive* residual is punching above its weight — often a well-coached rebuilding squad.

In [33]:
coach_tenures = [
    # Atlanta Hawks
    {'coach': 'Lenny Wilkens',    'team': 'Atlanta Hawks',          'start': '1999-00', 'end': '2004-05'},
    {'coach': 'Mike Woodson',     'team': 'Atlanta Hawks',          'start': '2004-05', 'end': '2011-12'},
    {'coach': 'Larry Drew',       'team': 'Atlanta Hawks',          'start': '2011-12', 'end': '2012-13'},
    {'coach': 'Mike Budenholzer', 'team': 'Atlanta Hawks',          'start': '2013-14', 'end': '2017-18'},
    {'coach': 'Lloyd Pierce',     'team': 'Atlanta Hawks',          'start': '2018-19', 'end': '2020-21'},
    {'coach': 'Nate McMillan',    'team': 'Atlanta Hawks',          'start': '2020-21', 'end': '2021-22'},
    {'coach': 'Quin Snyder',      'team': 'Atlanta Hawks',          'start': '2022-23', 'end': '2025-26'},

    # Boston Celtics
    {'coach': 'Rick Pitino',      'team': 'Boston Celtics',         'start': '1997-98', 'end': '2000-01'},
    {'coach': 'Jim O\'Brien',     'team': 'Boston Celtics',         'start': '2000-01', 'end': '2003-04'},
    {'coach': 'Doc Rivers',       'team': 'Boston Celtics',         'start': '2004-05', 'end': '2012-13'},
    {'coach': 'Brad Stevens',     'team': 'Boston Celtics',         'start': '2013-14', 'end': '2020-21'},
    {'coach': 'Ime Udoka',        'team': 'Boston Celtics',         'start': '2021-22', 'end': '2021-22'},
    {'coach': 'Joe Mazzulla',     'team': 'Boston Celtics',         'start': '2022-23', 'end': '2025-26'},

    # Brooklyn Nets / New Jersey Nets
    {'coach': 'Don Casey',        'team': 'New Jersey Nets',        'start': '1999-00', 'end': '2000-01'},
    {'coach': 'Byron Scott',      'team': 'New Jersey Nets',        'start': '2000-01', 'end': '2003-04'},
    {'coach': 'Lawrence Frank',   'team': 'New Jersey Nets',        'start': '2003-04', 'end': '2009-10'},
    {'coach': 'Avery Johnson',    'team': 'New Jersey Nets',        'start': '2010-11', 'end': '2011-12'},
    {'coach': 'P.J. Carlesimo',   'team': 'Brooklyn Nets',          'start': '2012-13', 'end': '2012-13'},
    {'coach': 'Jason Kidd',       'team': 'Brooklyn Nets',          'start': '2013-14', 'end': '2013-14'},
    {'coach': 'Lionel Hollins',   'team': 'Brooklyn Nets',          'start': '2014-15', 'end': '2015-16'},
    {'coach': 'Kenny Atkinson',   'team': 'Brooklyn Nets',          'start': '2016-17', 'end': '2019-20'},
    {'coach': 'Steve Nash',       'team': 'Brooklyn Nets',          'start': '2020-21', 'end': '2022-23'},
    {'coach': 'Jacque Vaughn',    'team': 'Brooklyn Nets',          'start': '2022-23', 'end': '2023-24'},
    {'coach': 'Jordi Fernandez',  'team': 'Brooklyn Nets',          'start': '2024-25', 'end': '2025-26'},

    # Charlotte Hornets / Bobcats
    {'coach': 'Paul Silas',       'team': 'Charlotte Hornets',      'start': '1999-00', 'end': '2002-03'},
    {'coach': 'Bernie Bickerstaff','team': 'Charlotte Bobcats',     'start': '2004-05', 'end': '2006-07'},
    {'coach': 'Sam Vincent',      'team': 'Charlotte Bobcats',      'start': '2007-08', 'end': '2008-09'},
    {'coach': 'Larry Brown',      'team': 'Charlotte Bobcats',      'start': '2008-09', 'end': '2010-11'},
    {'coach': 'Paul Silas',       'team': 'Charlotte Bobcats',      'start': '2010-11', 'end': '2011-12'},
    {'coach': 'Steve Clifford',   'team': 'Charlotte Hornets',      'start': '2013-14', 'end': '2017-18'},
    {'coach': 'James Borrego',    'team': 'Charlotte Hornets',      'start': '2018-19', 'end': '2021-22'},
    {'coach': 'Steve Clifford',   'team': 'Charlotte Hornets',      'start': '2022-23', 'end': '2023-24'},
    {'coach': 'Charles Lee',      'team': 'Charlotte Hornets',      'start': '2024-25', 'end': '2025-26'},

    # Chicago Bulls
    {'coach': 'Phil Jackson',     'team': 'Chicago Bulls',          'start': '1999-00', 'end': '1999-00'},
    {'coach': 'Tim Floyd',        'team': 'Chicago Bulls',          'start': '1999-00', 'end': '2001-02'},
    {'coach': 'Bill Cartwright',  'team': 'Chicago Bulls',          'start': '2001-02', 'end': '2002-03'},
    {'coach': 'Scott Skiles',     'team': 'Chicago Bulls',          'start': '2003-04', 'end': '2006-07'},
    {'coach': 'Jim Boylan',       'team': 'Chicago Bulls',          'start': '2007-08', 'end': '2007-08'},
    {'coach': 'Vinny Del Negro',  'team': 'Chicago Bulls',          'start': '2008-09', 'end': '2009-10'},
    {'coach': 'Tom Thibodeau',    'team': 'Chicago Bulls',          'start': '2010-11', 'end': '2014-15'},
    {'coach': 'Fred Hoiberg',     'team': 'Chicago Bulls',          'start': '2015-16', 'end': '2018-19'},
    {'coach': 'Jim Boylan',       'team': 'Chicago Bulls',          'start': '2018-19', 'end': '2018-19'},
    {'coach': 'Billy Donovan',    'team': 'Chicago Bulls',          'start': '2020-21', 'end': '2025-26'},

    # Cleveland Cavaliers
    {'coach': 'Randy Wittman',    'team': 'Cleveland Cavaliers',    'start': '1999-00', 'end': '2000-01'},
    {'coach': 'John Lucas',       'team': 'Cleveland Cavaliers',    'start': '2001-02', 'end': '2002-03'},
    {'coach': 'Paul Silas',       'team': 'Cleveland Cavaliers',    'start': '2003-04', 'end': '2004-05'},
    {'coach': 'Brendan Malone',   'team': 'Cleveland Cavaliers',    'start': '2004-05', 'end': '2004-05'},
    {'coach': 'Mike Brown',       'team': 'Cleveland Cavaliers',    'start': '2005-06', 'end': '2009-10'},
    {'coach': 'Byron Scott',      'team': 'Cleveland Cavaliers',    'start': '2010-11', 'end': '2012-13'},
    {'coach': 'Mike Brown',       'team': 'Cleveland Cavaliers',    'start': '2013-14', 'end': '2013-14'},
    {'coach': 'David Blatt',      'team': 'Cleveland Cavaliers',    'start': '2014-15', 'end': '2015-16'},
    {'coach': 'Tyronn Lue',       'team': 'Cleveland Cavaliers',    'start': '2015-16', 'end': '2018-19'},
    {'coach': 'John Beilein',     'team': 'Cleveland Cavaliers',    'start': '2019-20', 'end': '2019-20'},
    {'coach': 'J.B. Bickerstaff', 'team': 'Cleveland Cavaliers',    'start': '2019-20', 'end': '2023-24'},
    {'coach': 'Kenny Atkinson',   'team': 'Cleveland Cavaliers',    'start': '2024-25', 'end': '2025-26'},

    # Dallas Mavericks
    {'coach': 'Don Nelson',       'team': 'Dallas Mavericks',       'start': '1999-00', 'end': '2004-05'},
    {'coach': 'Avery Johnson',    'team': 'Dallas Mavericks',       'start': '2004-05', 'end': '2007-08'},
    {'coach': 'Rick Carlisle',    'team': 'Dallas Mavericks',       'start': '2008-09', 'end': '2020-21'},
    {'coach': 'Jason Kidd',       'team': 'Dallas Mavericks',       'start': '2021-22', 'end': '2023-24'},
    {'coach': 'Nico Harrison',    'team': 'Dallas Mavericks',       'start': '2024-25', 'end': '2025-26'},

    # Denver Nuggets
    {'coach': 'Dan Issel',        'team': 'Denver Nuggets',         'start': '1999-00', 'end': '2000-01'},
    {'coach': 'Jeff Bzdelik',     'team': 'Denver Nuggets',         'start': '2002-03', 'end': '2004-05'},
    {'coach': 'George Karl',      'team': 'Denver Nuggets',         'start': '2004-05', 'end': '2012-13'},
    {'coach': 'Brian Shaw',       'team': 'Denver Nuggets',         'start': '2013-14', 'end': '2014-15'},
    {'coach': 'Michael Malone',   'team': 'Denver Nuggets',         'start': '2015-16', 'end': '2024-25'},
    {'coach': 'David Adelman',    'team': 'Denver Nuggets',         'start': '2024-25', 'end': '2025-26'},

    # Detroit Pistons
    {'coach': 'Alvin Gentry',     'team': 'Detroit Pistons',        'start': '1999-00', 'end': '2000-01'},
    {'coach': 'Rick Carlisle',    'team': 'Detroit Pistons',        'start': '2001-02', 'end': '2002-03'},
    {'coach': 'Larry Brown',      'team': 'Detroit Pistons',        'start': '2003-04', 'end': '2004-05'},
    {'coach': 'Flip Saunders',    'team': 'Detroit Pistons',        'start': '2005-06', 'end': '2007-08'},
    {'coach': 'Michael Curry',    'team': 'Detroit Pistons',        'start': '2008-09', 'end': '2008-09'},
    {'coach': 'John Kuester',     'team': 'Detroit Pistons',        'start': '2009-10', 'end': '2010-11'},
    {'coach': 'Lawrence Frank',   'team': 'Detroit Pistons',        'start': '2011-12', 'end': '2012-13'},
    {'coach': 'Mo Cheeks',        'team': 'Detroit Pistons',        'start': '2013-14', 'end': '2013-14'},
    {'coach': 'Stan Van Gundy',   'team': 'Detroit Pistons',        'start': '2014-15', 'end': '2017-18'},
    {'coach': 'Dwane Casey',      'team': 'Detroit Pistons',        'start': '2018-19', 'end': '2022-23'},
    {'coach': 'Monty Williams',   'team': 'Detroit Pistons',        'start': '2023-24', 'end': '2023-24'},
    {'coach': 'J.B. Bickerstaff', 'team': 'Detroit Pistons',        'start': '2024-25', 'end': '2025-26'},

    # Golden State Warriors
    {'coach': 'Dave Cowens',      'team': 'Golden State Warriors',  'start': '1999-00', 'end': '2000-01'},
    {'coach': 'Brian Winters',    'team': 'Golden State Warriors',  'start': '2000-01', 'end': '2001-02'},
    {'coach': 'Eric Musselman',   'team': 'Golden State Warriors',  'start': '2002-03', 'end': '2003-04'},
    {'coach': 'Mike Montgomery',  'team': 'Golden State Warriors',  'start': '2004-05', 'end': '2005-06'},
    {'coach': 'Don Nelson',       'team': 'Golden State Warriors',  'start': '2006-07', 'end': '2009-10'},
    {'coach': 'Keith Smart',      'team': 'Golden State Warriors',  'start': '2010-11', 'end': '2011-12'},
    {'coach': 'Mark Jackson',     'team': 'Golden State Warriors',  'start': '2011-12', 'end': '2013-14'},
    {'coach': 'Steve Kerr',       'team': 'Golden State Warriors',  'start': '2014-15', 'end': '2025-26'},

    # Houston Rockets
    {'coach': 'Rudy Tomjanovich', 'team': 'Houston Rockets',        'start': '1999-00', 'end': '2002-03'},
    {'coach': 'Jeff Van Gundy',   'team': 'Houston Rockets',        'start': '2003-04', 'end': '2006-07'},
    {'coach': 'Rick Adelman',     'team': 'Houston Rockets',        'start': '2007-08', 'end': '2010-11'},
    {'coach': 'Kevin McHale',     'team': 'Houston Rockets',        'start': '2011-12', 'end': '2014-15'},
    {'coach': 'J.B. Bickerstaff', 'team': 'Houston Rockets',        'start': '2015-16', 'end': '2015-16'},
    {'coach': 'Mike D\'Antoni',   'team': 'Houston Rockets',        'start': '2016-17', 'end': '2019-20'},
    {'coach': 'Stephen Silas',    'team': 'Houston Rockets',        'start': '2020-21', 'end': '2023-24'},
    {'coach': 'Ime Udoka',        'team': 'Houston Rockets',        'start': '2024-25', 'end': '2025-26'},

    # Indiana Pacers
    {'coach': 'Larry Bird',       'team': 'Indiana Pacers',         'start': '1997-98', 'end': '1999-00'},
    {'coach': 'Isiah Thomas',     'team': 'Indiana Pacers',         'start': '2000-01', 'end': '2002-03'},
    {'coach': 'Rick Carlisle',    'team': 'Indiana Pacers',         'start': '2003-04', 'end': '2006-07'},
    {'coach': 'Jim O\'Brien',     'team': 'Indiana Pacers',         'start': '2007-08', 'end': '2010-11'},
    {'coach': 'Frank Vogel',      'team': 'Indiana Pacers',         'start': '2010-11', 'end': '2015-16'},
    {'coach': 'Nate McMillan',    'team': 'Indiana Pacers',         'start': '2016-17', 'end': '2019-20'},
    {'coach': 'Nate Bjorkgren',   'team': 'Indiana Pacers',         'start': '2020-21', 'end': '2020-21'},
    {'coach': 'Rick Carlisle',    'team': 'Indiana Pacers',         'start': '2021-22', 'end': '2025-26'},

    # Los Angeles Clippers
    {'coach': 'Alvin Gentry',     'team': 'Los Angeles Clippers',   'start': '2000-01', 'end': '2002-03'},
    {'coach': 'Mike Dunleavy',    'team': 'Los Angeles Clippers',   'start': '2003-04', 'end': '2009-10'},
    {'coach': 'Vinny Del Negro',  'team': 'Los Angeles Clippers',   'start': '2010-11', 'end': '2012-13'},
    {'coach': 'Doc Rivers',       'team': 'Los Angeles Clippers',   'start': '2013-14', 'end': '2019-20'},
    {'coach': 'Tyronn Lue',       'team': 'Los Angeles Clippers',   'start': '2020-21', 'end': '2025-26'},

    # Los Angeles Lakers
    {'coach': 'Phil Jackson',     'team': 'Los Angeles Lakers',     'start': '1999-00', 'end': '2003-04'},
    {'coach': 'Rudy Tomjanovich', 'team': 'Los Angeles Lakers',     'start': '2004-05', 'end': '2004-05'},
    {'coach': 'Phil Jackson',     'team': 'Los Angeles Lakers',     'start': '2005-06', 'end': '2010-11'},
    {'coach': 'Mike Brown',       'team': 'Los Angeles Lakers',     'start': '2011-12', 'end': '2012-13'},
    {'coach': 'Mike D\'Antoni',   'team': 'Los Angeles Lakers',     'start': '2012-13', 'end': '2013-14'},
    {'coach': 'Byron Scott',      'team': 'Los Angeles Lakers',     'start': '2014-15', 'end': '2015-16'},
    {'coach': 'Luke Walton',      'team': 'Los Angeles Lakers',     'start': '2016-17', 'end': '2018-19'},
    {'coach': 'Frank Vogel',      'team': 'Los Angeles Lakers',     'start': '2019-20', 'end': '2021-22'},
    {'coach': 'Darvin Ham',       'team': 'Los Angeles Lakers',     'start': '2022-23', 'end': '2023-24'},
    {'coach': 'JJ Redick',        'team': 'Los Angeles Lakers',     'start': '2024-25', 'end': '2025-26'},

    # Memphis Grizzlies
    {'coach': 'Sidney Lowe',      'team': 'Memphis Grizzlies',      'start': '1999-00', 'end': '2000-01'},
    {'coach': 'Hubie Brown',      'team': 'Memphis Grizzlies',      'start': '2002-03', 'end': '2004-05'},
    {'coach': 'Mike Fratello',    'team': 'Memphis Grizzlies',      'start': '2004-05', 'end': '2006-07'},
    {'coach': 'Marc Iavaroni',    'team': 'Memphis Grizzlies',      'start': '2007-08', 'end': '2008-09'},
    {'coach': 'Lionel Hollins',   'team': 'Memphis Grizzlies',      'start': '2009-10', 'end': '2012-13'},
    {'coach': 'Dave Joerger',     'team': 'Memphis Grizzlies',      'start': '2013-14', 'end': '2015-16'},
    {'coach': 'David Fizdale',    'team': 'Memphis Grizzlies',      'start': '2016-17', 'end': '2017-18'},
    {'coach': 'J.B. Bickerstaff', 'team': 'Memphis Grizzlies',      'start': '2018-19', 'end': '2018-19'},
    {'coach': 'Taylor Jenkins',   'team': 'Memphis Grizzlies',      'start': '2019-20', 'end': '2024-25'},
    {'coach': 'Tuomas Iisalo',    'team': 'Memphis Grizzlies',      'start': '2024-25', 'end': '2025-26'},

    # Miami Heat
    {'coach': 'Pat Riley',        'team': 'Miami Heat',             'start': '1999-00', 'end': '2002-03'},
    {'coach': 'Stan Van Gundy',   'team': 'Miami Heat',             'start': '2003-04', 'end': '2005-06'},
    {'coach': 'Pat Riley',        'team': 'Miami Heat',             'start': '2005-06', 'end': '2007-08'},
    {'coach': 'Erik Spoelstra',   'team': 'Miami Heat',             'start': '2008-09', 'end': '2025-26'},

    # Milwaukee Bucks
    {'coach': 'George Karl',      'team': 'Milwaukee Bucks',        'start': '1998-99', 'end': '2002-03'},
    {'coach': 'Terry Porter',     'team': 'Milwaukee Bucks',        'start': '2003-04', 'end': '2004-05'},
    {'coach': 'Terry Stotts',     'team': 'Milwaukee Bucks',        'start': '2005-06', 'end': '2006-07'},
    {'coach': 'Larry Krystkowiak','team': 'Milwaukee Bucks',        'start': '2006-07', 'end': '2007-08'},
    {'coach': 'Scott Skiles',     'team': 'Milwaukee Bucks',        'start': '2008-09', 'end': '2012-13'},
    {'coach': 'Larry Drew',       'team': 'Milwaukee Bucks',        'start': '2013-14', 'end': '2013-14'},
    {'coach': 'Jason Kidd',       'team': 'Milwaukee Bucks',        'start': '2014-15', 'end': '2017-18'},
    {'coach': 'Joe Prunty',       'team': 'Milwaukee Bucks',        'start': '2017-18', 'end': '2017-18'},
    {'coach': 'Mike Budenholzer', 'team': 'Milwaukee Bucks',        'start': '2018-19', 'end': '2022-23'},
    {'coach': 'Adrian Griffin',   'team': 'Milwaukee Bucks',        'start': '2023-24', 'end': '2023-24'},
    {'coach': 'Doc Rivers',       'team': 'Milwaukee Bucks',        'start': '2023-24', 'end': '2025-26'},

    # Minnesota Timberwolves
    {'coach': 'Flip Saunders',    'team': 'Minnesota Timberwolves', 'start': '1999-00', 'end': '2004-05'},
    {'coach': 'Dwane Casey',      'team': 'Minnesota Timberwolves', 'start': '2005-06', 'end': '2006-07'},
    {'coach': 'Randy Wittman',    'team': 'Minnesota Timberwolves', 'start': '2007-08', 'end': '2008-09'},
    {'coach': 'Kurt Rambis',      'team': 'Minnesota Timberwolves', 'start': '2009-10', 'end': '2010-11'},
    {'coach': 'Rick Adelman',     'team': 'Minnesota Timberwolves', 'start': '2011-12', 'end': '2013-14'},
    {'coach': 'Flip Saunders',    'team': 'Minnesota Timberwolves', 'start': '2014-15', 'end': '2015-16'},
    {'coach': 'Tom Thibodeau',    'team': 'Minnesota Timberwolves', 'start': '2016-17', 'end': '2018-19'},
    {'coach': 'Ryan Saunders',    'team': 'Minnesota Timberwolves', 'start': '2018-19', 'end': '2020-21'},
    {'coach': 'Chris Finch',      'team': 'Minnesota Timberwolves', 'start': '2020-21', 'end': '2025-26'},

    # New Orleans Pelicans / Hornets
    {'coach': 'Paul Silas',       'team': 'New Orleans Hornets',    'start': '2002-03', 'end': '2004-05'},
    {'coach': 'Byron Scott',      'team': 'New Orleans Hornets',    'start': '2004-05', 'end': '2009-10'},
    {'coach': 'Monty Williams',   'team': 'New Orleans Hornets',    'start': '2010-11', 'end': '2014-15'},
    {'coach': 'Alvin Gentry',     'team': 'New Orleans Pelicans',   'start': '2015-16', 'end': '2018-19'},
    {'coach': 'Monty Williams',   'team': 'New Orleans Pelicans',   'start': '2019-20', 'end': '2019-20'},
    {'coach': 'Stan Van Gundy',   'team': 'New Orleans Pelicans',   'start': '2020-21', 'end': '2020-21'},
    {'coach': 'Willie Green',     'team': 'New Orleans Pelicans',   'start': '2021-22', 'end': '2025-26'},
    {'coach': 'James Borrego',    'team': 'New Orleans Pelicans',   'start': '2025-26', 'end': '2025-26'},

    # New York Knicks
    {'coach': 'Jeff Van Gundy',   'team': 'New York Knicks',        'start': '1999-00', 'end': '2001-02'},
    {'coach': 'Don Chaney',       'team': 'New York Knicks',        'start': '2001-02', 'end': '2003-04'},
    {'coach': 'Lenny Wilkens',    'team': 'New York Knicks',        'start': '2004-05', 'end': '2004-05'},
    {'coach': 'Larry Brown',      'team': 'New York Knicks',        'start': '2005-06', 'end': '2005-06'},
    {'coach': 'Isiah Thomas',     'team': 'New York Knicks',        'start': '2006-07', 'end': '2007-08'},
    {'coach': 'Mike D\'Antoni',   'team': 'New York Knicks',        'start': '2008-09', 'end': '2011-12'},
    {'coach': 'Mike Woodson',     'team': 'New York Knicks',        'start': '2011-12', 'end': '2013-14'},
    {'coach': 'Derek Fisher',     'team': 'New York Knicks',        'start': '2014-15', 'end': '2015-16'},
    {'coach': 'Jeff Hornacek',    'team': 'New York Knicks',        'start': '2016-17', 'end': '2017-18'},
    {'coach': 'David Fizdale',    'team': 'New York Knicks',        'start': '2018-19', 'end': '2019-20'},
    {'coach': 'Tom Thibodeau',    'team': 'New York Knicks',        'start': '2020-21', 'end': '2024-25'},
    {'coach': 'Mike Brown',       'team': 'New York Knicks',        'start': '2025-26', 'end': '2025-26'},

    # Oklahoma City Thunder / Seattle SuperSonics
    {'coach': 'Paul Westphal',    'team': 'Seattle SuperSonics',    'start': '1999-00', 'end': '2000-01'},
    {'coach': 'Nate McMillan',    'team': 'Seattle SuperSonics',    'start': '2000-01', 'end': '2004-05'},
    {'coach': 'Bob Hill',         'team': 'Seattle SuperSonics',    'start': '2005-06', 'end': '2006-07'},
    {'coach': 'P.J. Carlesimo',   'team': 'Seattle SuperSonics',    'start': '2007-08', 'end': '2007-08'},
    {'coach': 'Scott Brooks',     'team': 'Oklahoma City Thunder',  'start': '2008-09', 'end': '2014-15'},
    {'coach': 'Billy Donovan',    'team': 'Oklahoma City Thunder',  'start': '2015-16', 'end': '2019-20'},
    {'coach': 'Mark Daigneault',  'team': 'Oklahoma City Thunder',  'start': '2020-21', 'end': '2025-26'},

    # Orlando Magic
    {'coach': 'Doc Rivers',       'team': 'Orlando Magic',          'start': '1999-00', 'end': '2003-04'},
    {'coach': 'Johnny Davis',     'team': 'Orlando Magic',          'start': '2003-04', 'end': '2004-05'},
    {'coach': 'Brian Hill',       'team': 'Orlando Magic',          'start': '2005-06', 'end': '2006-07'},
    {'coach': 'Stan Van Gundy',   'team': 'Orlando Magic',          'start': '2007-08', 'end': '2011-12'},
    {'coach': 'Jacque Vaughn',    'team': 'Orlando Magic',          'start': '2012-13', 'end': '2014-15'},
    {'coach': 'Scott Skiles',     'team': 'Orlando Magic',          'start': '2015-16', 'end': '2015-16'},
    {'coach': 'Frank Vogel',      'team': 'Orlando Magic',          'start': '2015-16', 'end': '2017-18'},
    {'coach': 'Steve Clifford',   'team': 'Orlando Magic',          'start': '2018-19', 'end': '2020-21'},
    {'coach': 'Jamahl Mosley',    'team': 'Orlando Magic',          'start': '2021-22', 'end': '2025-26'},

    # Philadelphia 76ers
    {'coach': 'Larry Brown',      'team': 'Philadelphia 76ers',     'start': '1997-98', 'end': '2002-03'},
    {'coach': 'Randy Ayers',      'team': 'Philadelphia 76ers',     'start': '2003-04', 'end': '2003-04'},
    {'coach': 'Jim O\'Brien',     'team': 'Philadelphia 76ers',     'start': '2004-05', 'end': '2007-08'},
    {'coach': 'Tony DiLeo',       'team': 'Philadelphia 76ers',     'start': '2008-09', 'end': '2008-09'},
    {'coach': 'Eddie Jordan',     'team': 'Philadelphia 76ers',     'start': '2009-10', 'end': '2009-10'},
    {'coach': 'Doug Collins',     'team': 'Philadelphia 76ers',     'start': '2010-11', 'end': '2012-13'},
    {'coach': 'Brett Brown',      'team': 'Philadelphia 76ers',     'start': '2013-14', 'end': '2019-20'},
    {'coach': 'Doc Rivers',       'team': 'Philadelphia 76ers',     'start': '2020-21', 'end': '2022-23'},
    {'coach': 'Nick Nurse',       'team': 'Philadelphia 76ers',     'start': '2023-24', 'end': '2025-26'},

    # Phoenix Suns
    {'coach': 'Scott Skiles',     'team': 'Phoenix Suns',           'start': '1999-00', 'end': '2000-01'},
    {'coach': 'Frank Johnson',    'team': 'Phoenix Suns',           'start': '2002-03', 'end': '2003-04'},
    {'coach': 'Mike D\'Antoni',   'team': 'Phoenix Suns',           'start': '2003-04', 'end': '2007-08'},
    {'coach': 'Terry Porter',     'team': 'Phoenix Suns',           'start': '2008-09', 'end': '2008-09'},
    {'coach': 'Alvin Gentry',     'team': 'Phoenix Suns',           'start': '2009-10', 'end': '2012-13'},
    {'coach': 'Jeff Hornacek',    'team': 'Phoenix Suns',           'start': '2013-14', 'end': '2015-16'},
    {'coach': 'Earl Watson',      'team': 'Phoenix Suns',           'start': '2016-17', 'end': '2017-18'},
    {'coach': 'Igor Kokoskov',    'team': 'Phoenix Suns',           'start': '2018-19', 'end': '2018-19'},
    {'coach': 'Monty Williams',   'team': 'Phoenix Suns',           'start': '2019-20', 'end': '2022-23'},
    {'coach': 'Frank Vogel',      'team': 'Phoenix Suns',           'start': '2023-24', 'end': '2023-24'},
    {'coach': 'Mike Budenholzer', 'team': 'Phoenix Suns',           'start': '2024-25', 'end': '2024-25'},
    {'coach': 'Jordan Ott',       'team': 'Phoenix Suns',           'start': '2025-26', 'end': '2025-26'},

    # Portland Trail Blazers
    {'coach': 'Mike Dunleavy',    'team': 'Portland Trail Blazers', 'start': '1997-98', 'end': '2000-01'},
    {'coach': 'Maurice Cheeks',   'team': 'Portland Trail Blazers', 'start': '2001-02', 'end': '2004-05'},
    {'coach': 'Nate McMillan',    'team': 'Portland Trail Blazers', 'start': '2005-06', 'end': '2011-12'},
    {'coach': 'Terry Stotts',     'team': 'Portland Trail Blazers', 'start': '2012-13', 'end': '2020-21'},
    {'coach': 'Chauncey Billups', 'team': 'Portland Trail Blazers', 'start': '2021-22', 'end': '2025-26'},

    # Sacramento Kings
    {'coach': 'Rick Adelman',     'team': 'Sacramento Kings',       'start': '1999-00', 'end': '2005-06'},
    {'coach': 'Eric Musselman',   'team': 'Sacramento Kings',       'start': '2006-07', 'end': '2006-07'},
    {'coach': 'Reggie Theus',     'team': 'Sacramento Kings',       'start': '2007-08', 'end': '2008-09'},
    {'coach': 'Kenny Natt',       'team': 'Sacramento Kings',       'start': '2008-09', 'end': '2009-10'},
    {'coach': 'Paul Westphal',    'team': 'Sacramento Kings',       'start': '2009-10', 'end': '2010-11'},
    {'coach': 'Keith Smart',      'team': 'Sacramento Kings',       'start': '2011-12', 'end': '2011-12'},
    {'coach': 'Michael Malone',   'team': 'Sacramento Kings',       'start': '2013-14', 'end': '2014-15'},
    {'coach': 'George Karl',      'team': 'Sacramento Kings',       'start': '2015-16', 'end': '2015-16'},
    {'coach': 'Dave Joerger',     'team': 'Sacramento Kings',       'start': '2016-17', 'end': '2018-19'},
    {'coach': 'Luke Walton',      'team': 'Sacramento Kings',       'start': '2019-20', 'end': '2021-22'},
    {'coach': 'Mike Brown',       'team': 'Sacramento Kings',       'start': '2022-23', 'end': '2024-25'},
    {'coach': 'Doug Christie',    'team': 'Sacramento Kings',       'start': '2024-25', 'end': '2025-26'},

    # San Antonio Spurs
    {'coach': 'Gregg Popovich',   'team': 'San Antonio Spurs',      'start': '1999-00', 'end': '2024-25'},
    {'coach': 'Mitch Johnson',    'team': 'San Antonio Spurs',      'start': '2024-25', 'end': '2025-26'},

    # Toronto Raptors
    {'coach': 'Butch Carter',     'team': 'Toronto Raptors',        'start': '1999-00', 'end': '2000-01'},
    {'coach': 'Lenny Wilkens',    'team': 'Toronto Raptors',        'start': '2000-01', 'end': '2002-03'},
    {'coach': 'Kevin O\'Neill',   'team': 'Toronto Raptors',        'start': '2003-04', 'end': '2003-04'},
    {'coach': 'Sam Mitchell',     'team': 'Toronto Raptors',        'start': '2004-05', 'end': '2007-08'},
    {'coach': 'Jay Triano',       'team': 'Toronto Raptors',        'start': '2008-09', 'end': '2010-11'},
    {'coach': 'Dwane Casey',      'team': 'Toronto Raptors',        'start': '2011-12', 'end': '2017-18'},
    {'coach': 'Nick Nurse',       'team': 'Toronto Raptors',        'start': '2018-19', 'end': '2022-23'},
    {'coach': 'Darko Rajakovic',  'team': 'Toronto Raptors',        'start': '2023-24', 'end': '2025-26'},

    # Utah Jazz
    {'coach': 'Jerry Sloan',      'team': 'Utah Jazz',              'start': '1999-00', 'end': '2010-11'},
    {'coach': 'Ty Corbin',        'team': 'Utah Jazz',              'start': '2010-11', 'end': '2013-14'},
    {'coach': 'Quin Snyder',      'team': 'Utah Jazz',              'start': '2014-15', 'end': '2021-22'},
    {'coach': 'Will Hardy',       'team': 'Utah Jazz',              'start': '2022-23', 'end': '2025-26'},

    # Washington Wizards
    {'coach': 'Gar Heard',        'team': 'Washington Wizards',     'start': '1999-00', 'end': '2000-01'},
    {'coach': 'Doug Collins',     'team': 'Washington Wizards',     'start': '2001-02', 'end': '2002-03'},
    {'coach': 'Eddie Jordan',     'team': 'Washington Wizards',     'start': '2003-04', 'end': '2008-09'},
    {'coach': 'Flip Saunders',    'team': 'Washington Wizards',     'start': '2009-10', 'end': '2011-12'},
    {'coach': 'Randy Wittman',    'team': 'Washington Wizards',     'start': '2011-12', 'end': '2015-16'},
    {'coach': 'Scott Brooks',     'team': 'Washington Wizards',     'start': '2016-17', 'end': '2020-21'},
    {'coach': 'Wes Unseld Jr.',   'team': 'Washington Wizards',     'start': '2021-22', 'end': '2023-24'},
    {'coach': 'Brian Keefe',      'team': 'Washington Wizards',     'start': '2023-24', 'end': '2025-26'},
]

coach_df = pd.DataFrame(coach_tenures)

def add_residual_vs_era(d):
    d = d.copy()
    d['residual_vs_era'] = d.groupby('SEASON')['residual'].transform(lambda x: x - x.mean())
    return d

# ── All-Seasons Model (used for coaching analysis) ───────────────────────────
# This model trains on all 30 seasons at once — NOT walk-forward.
# Every coach is judged by the same baseline regardless of era.
# Trade-off: the model has seen the data it predicts, so residuals reflect
# consistency relative to peers rather than out-of-sample surprise.

df_raw = df.copy()  # df already has all seasons from the walk-forward step

X_all = df_raw[base_feats].apply(pd.to_numeric, errors='coerce').dropna()
y_all = df_raw.loc[X_all.index, 'W_PCT_scaled']

model_full = RidgeCV(alphas=[0.01, 0.05, 0.1, 0.5, 1.0, 5.0, 10.0], cv=5).fit(X_all, y_all)
y_pred_full = model_full.predict(X_all)

df_full = df_raw.loc[X_all.index].copy()
df_full['residual'] = y_all.values - y_pred_full
df_full = add_residual_vs_era(df_full)

print(f"All-seasons model ready — alpha: {model_full.alpha_}, rows: {len(df_full)}")

All-seasons model ready — alpha: 0.01, rows: 805


## Coach Tenure Data & All-Seasons Model

The cell above defines every head coaching tenure for all 30 NBA franchises from 1996–97 through 2025–26. Each entry stores the coach's name, their team, and the first and last season of that stint. This list is converted to a DataFrame (`coach_df`) at the bottom of the cell.

---

## Why a Separate Model for Coaching?

The team charts use walk-forward validation, which is the most *honest* way to predict each season — but it means early seasons are predicted with very little training data (only 3–4 seasons), which makes their residuals noisier and harder to compare directly to residuals from later seasons trained on 20+ years of data.

For coaching analysis the goal is different: we want to judge every coach by the *same standard* regardless of when they coached. So the coaching model trains on all 30 seasons at once. This gives every prediction the same baseline, making Jerry Sloan's 2003 Jazz directly comparable to Gregg Popovich's 2023 Spurs.

The trade-off is that this model has seen the data it is predicting — so these residuals measure *consistency relative to peers*, not genuine out-of-sample surprise. That distinction is noted in the output.

In [45]:
# match each season in df_full to a coach based on team and season range
def get_coach(row):
    for _, coach in coach_df.iterrows():
        if row['TEAM_NAME'] == coach['team'] and coach['start'] <= row['SEASON'] <= coach['end']:
            return coach['coach']
    return 'Other'

# df_full is a separate version of the dataset trained on all seasons at once rather than 
# just past seasons — this gives a fairer comparison for coaches across different eras
# since every season is predicted with the same amount of information
df_full['coach'] = df_full.apply(get_coach, axis=1)

# calculate average residual per coach with tenure info
coach_residuals = df_full[df_full['coach'] != 'Other'].groupby(['coach', 'TEAM_NAME']).agg(
    avg_residual=('residual', 'mean'),
    seasons_coached=('residual', 'count'),
    start_season=('SEASON', 'min'),
    end_season=('SEASON', 'max')
)

# round only numeric columns
coach_residuals['avg_residual'] = coach_residuals['avg_residual'].round(3)
coach_residuals = coach_residuals[coach_residuals['seasons_coached'] >= 3]

# subtract mean so results are relative to average coach
coach_residuals['avg_residual'] = coach_residuals['avg_residual'] - coach_residuals['avg_residual'].mean()

# rank by percentile
coach_residuals['percentile'] = coach_residuals['avg_residual'].rank(pct=True).mul(100).round(1)
coach_residuals = coach_residuals.sort_values('avg_residual', ascending=False)

print(coach_residuals[['avg_residual', 'seasons_coached', 'start_season', 'end_season', 'percentile']].to_string())

                                           avg_residual  seasons_coached start_season end_season  percentile
coach              TEAM_NAME                                                                                
Mike D'Antoni      Houston Rockets              1.12637                4      2016-17    2019-20       100.0
Jason Kidd         Dallas Mavericks             0.56937                3      2021-22    2023-24        99.2
Chauncey Billups   Portland Trail Blazers       0.53337                5      2021-22    2025-26        98.3
Stan Van Gundy     Orlando Magic                0.43837                5      2007-08    2011-12        97.5
Tyronn Lue         Cleveland Cavaliers          0.41037                3      2016-17    2018-19        96.6
Kevin McHale       Houston Rockets              0.37837                4      2011-12    2014-15        95.8
Jim O'Brien        Boston Celtics               0.32537                3      2001-02    2003-04        95.0
Joe Mazzulla       

## League-Wide Coaching Rankings

The table above ranks every head coach in the dataset (minimum 3 seasons) by their **average residual relative to the league mean**. Zero represents the average NBA coach. A positive number means that coach consistently got more wins than their team's stats predicted; a negative number means the opposite.

**Columns explained:**

| Column | Meaning |
|---|---|
| `avg_residual` | Average gap between actual and predicted wins (scaled), relative to the league mean. Higher = better. |
| `seasons_coached` | How many seasons are in the sample. More seasons = more reliable signal. |
| `start_season` / `end_season` | The range of seasons covered by this tenure. |
| `percentile` | Where this coach ranks across all coaches in the dataset. 90 = top 10%. |

### League Coaching Rankings Table
- **High percentile coaches** are extracting wins the model did not expect. Focus on coaches with many seasons in the sample — a 3-season sample is meaningful, but a 10-season track record is far more reliable.
- **Near-zero coaches** are not necessarily bad — they may simply coach rosters so talented that the model already expected them to win. Context matters.

**What this measures and what it doesn't.** A high ranking means the coach consistently extracted *more wins than the roster's raw stats suggested*. It does not mean they are the "best coach" in an absolute sense — a hall-of-fame coach on a hall-of-fame roster may rank near zero simply because the model already expected the team to win. The coaches who stand out most here are the ones who consistently got more out of less.

The minimum of 3 seasons filters out mid-season interim coaches whose small sample sizes would produce extreme (and meaningless) residuals in either direction.

In [43]:
# ── Interactive Team Coaching Chart ──────────────────────────────────────────
ALL_TEAMS_FULL = sorted(df_full[df_full['coach'] != 'Other']['TEAM_NAME'].unique().tolist())

team_coaching_dropdown = widgets.Dropdown(
    options=ALL_TEAMS_FULL,
    value='Houston Rockets',
    description='Team:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px')
)

coaching_output = widgets.Output()

def plot_team_coaching(team_filter):
    team_coaches = df_full[
        (df_full['coach'] != 'Other') & (df_full['TEAM_NAME'] == team_filter)
    ].groupby('coach').agg(
        avg_residual=('residual', 'mean'),
        seasons_coached=('residual', 'count'),
        start_season=('SEASON', 'min'),
        end_season=('SEASON', 'max')
    )
    team_coaches['avg_residual'] = team_coaches['avg_residual'].round(3)

    if team_coaches.empty:
        print(f"No coaching data for {team_filter}")
        return

    team_coaches['avg_residual_vs_team'] = team_coaches['avg_residual'] - team_coaches['avg_residual'].mean()
    team_coaches_team = team_coaches.sort_values('avg_residual_vs_team', ascending=False)

    team_coaches['avg_residual_vs_league'] = (team_coaches['avg_residual'] - coach_residuals['avg_residual'].mean()).round(3)
    team_coaches['percentile'] = team_coaches['avg_residual_vs_league'].rank(pct=True).mul(100).round(1)
    team_coaches_league = team_coaches.sort_values('avg_residual_vs_league', ascending=False)
    print(f"\n{team_filter} — All Coaches vs League Average:")
    print(team_coaches_league[['avg_residual_vs_league', 'seasons_coached', 'start_season', 'end_season']].to_string())

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    colors_team = ['green' if r > 0 else 'red' for r in team_coaches_team['avg_residual_vs_team']]
    axes[0].bar(team_coaches_team.index, team_coaches_team['avg_residual_vs_team'], color=colors_team)
    axes[0].axhline(y=0, color='black', linewidth=0.8)
    axes[0].set_xticks(range(len(team_coaches_team)))
    axes[0].set_xticklabels(team_coaches_team.index, rotation=45, ha='right', fontsize=8)
    axes[0].set_ylabel('Residual vs Team Average')
    axes[0].set_title(f'{team_filter} — Coaching Impact vs Team Average')

    colors_league = ['green' if r > 0 else 'red' for r in team_coaches_league['avg_residual_vs_league']]
    axes[1].bar(team_coaches_league.index, team_coaches_league['avg_residual_vs_league'], color=colors_league)
    axes[1].axhline(y=0, color='black', linewidth=0.8)
    axes[1].set_xticks(range(len(team_coaches_league)))
    axes[1].set_xticklabels(team_coaches_league.index, rotation=45, ha='right', fontsize=8)
    axes[1].set_ylabel('Residual vs League Average')
    axes[1].set_title(f'{team_filter} — Coaching Impact vs League Average')

    plt.tight_layout()
    plt.show()

def on_coaching_change(change):
    coaching_output.clear_output(wait=True)
    with coaching_output:
        plot_team_coaching(change['new'])

team_coaching_dropdown.observe(on_coaching_change, names='value')

display(widgets.VBox([
    widgets.Label('Select a team to view its coaching history:'),
    team_coaching_dropdown,
    coaching_output
]))

with coaching_output:
    plot_team_coaching(team_coaching_dropdown.value)

## Team Coaching History

The interactive charts above let you drill into a single franchise. Use the dropdown to select any team and see two side-by-side bar charts.

**Left chart — vs team average.** Each bar shows a coach's residual relative to the franchise's own historical coaching baseline. This answers: *"Was this coach unusually good or bad for this specific organisation?"* Green = above the team's average; red = below it.

**Right chart — vs league average.** The same coaches, now benchmarked against all NBA coaches. This answers: *"How did this franchise's coaches compare to the rest of the league?"* A coach can look great internally but still be below the league average, or vice versa.

In [41]:
# ── Interactive Coach Career Chart ───────────────────────────────────────────
ALL_COACHES = sorted(df_full[df_full['coach'] != 'Other']['coach'].unique().tolist())

coach_dropdown = widgets.Dropdown(
    options=ALL_COACHES,
    value="Mike D'Antoni",
    description='Coach:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px')
)

season_start_widget = widgets.Dropdown(
    options=seasons,
    value='1999-00',
    description='From:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='180px')
)

season_end_widget = widgets.Dropdown(
    options=seasons,
    value='2025-26',
    description='To:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='180px')
)

def plot_coach_career(coach_filter, season_start, season_end):
    tenures = coach_df[coach_df['coach'] == coach_filter]

    coach_seasons = df_full[df_full.apply(lambda row: any(
        row['coach'] == coach_filter and
        row['SEASON'] >= t['start'] and
        row['SEASON'] <= t['end']
        for _, t in tenures.iterrows()
    ), axis=1)][['SEASON', 'TEAM_NAME', 'coach', 'residual']].copy()

    coach_seasons = coach_seasons[
        (coach_seasons['SEASON'] >= season_start) &
        (coach_seasons['SEASON'] <= season_end)
    ]

    if coach_seasons.empty:
        print(f"No seasons found for {coach_filter} in the selected range.")
        return

    league_mean = df_full[df_full['SEASON'].between(season_start, season_end)]['residual'].mean()
    coach_seasons['residual_vs_league'] = (coach_seasons['residual'] - league_mean).round(3)
    coach_seasons = coach_seasons.sort_values('SEASON')

    print(f"\n{coach_filter} ({season_start} to {season_end}):")
    print(coach_seasons[['SEASON', 'TEAM_NAME', 'residual_vs_league']].to_string(index=False))

    fig, ax = plt.subplots(figsize=(14, 6))
    colors = ['green' if r > 0 else 'red' for r in coach_seasons['residual_vs_league']]
    x_labels = coach_seasons['SEASON'] + '\n' + coach_seasons['TEAM_NAME']
    ax.bar(x_labels, coach_seasons['residual_vs_league'], color=colors)
    ax.axhline(y=0, color='black', linewidth=0.8)
    ax.set_xticks(range(len(x_labels)))
    ax.set_xticklabels(x_labels, rotation=90, fontsize=7)
    ax.set_ylabel('Residual vs League Average')
    ax.set_title(f'{coach_filter} — Season-by-Season Coaching Impact vs League Average')
    plt.tight_layout()
    display(fig)
    plt.close(fig)

coach_out = widgets.interactive_output(
    plot_coach_career,
    {'coach_filter': coach_dropdown, 'season_start': season_start_widget, 'season_end': season_end_widget}
)
display(
    widgets.VBox([
        widgets.Label('Select a coach and date range:'),
        widgets.HBox([coach_dropdown, season_start_widget, season_end_widget])
    ]),
    coach_out
)

Output()

## Coach Career Chart

The interactive chart above shows a single coach's season-by-season residual across their entire career. Use the dropdown to select any coach, and the date range selectors to zoom in on a specific period.

Each bar represents one season. The label on the x-axis shows both the season and the team, so coaching stints at multiple franchises are easy to distinguish. Green = above league average that season; red = below it. The black horizontal line at zero is the league mean.

This view answers: *"Did this coach's impact change over time, and did it travel with them to different teams?"* A coach whose bars are consistently green across multiple franchises has a track record that is hard to attribute to roster luck. A coach who looks great at one stop but poor at another may have benefited from a specific organisational context or roster fit.

---

## A Note on All Three Coaching Views

All three coaching outputs — the league table, the team charts, and the career chart — use the all-seasons model rather than the walk-forward model. This is a deliberate choice so that every coach is evaluated on the same baseline regardless of era. The residuals here reflect *consistency relative to peers*, not out-of-sample predictive surprise.

Factors this model cannot see: injuries to key players, trades, tanking, schedule strength, and mid-season roster changes. A coach who inherits a depleted roster will look worse than they should; a coach who gains a superstar mid-season will look better. Use these numbers as a starting point for a conversation, not the final word.

In [116]:
df_full[['SEASON', 'TEAM_NAME', 'coach', 'W', 'W_PCT', 'residual_vs_era']].sort_values('residual_vs_era', ascending=False).head(30)

,SEASON,TEAM_NAME,coach,W,W_PCT,residual_vs_era
632,2017-18,Houston Rockets,Mike D'Antoni,65,0.792683,1.305368
662,2018-19,Houston Rockets,Mike D'Antoni,53,0.646341,1.221273
542,2014-15,Houston Rockets,Kevin McHale,56,0.682927,1.141978
692,2019-20,Houston Rockets,Mike D'Antoni,44,0.611111,1.102585
602,2016-17,Houston Rockets,Mike D'Antoni,55,0.670732,1.093569
13,1996-97,Miami Heat,Other,61,0.743902,0.993576
76,1998-99,Orlando Magic,Other,33,0.660000,0.862353
18,1996-97,Orlando Magic,Other,45,0.548780,0.852829
627,2017-18,Cleveland Cavaliers,Tyronn Lue,50,0.609756,0.842340
886,2025-26,Portland Trail Blazers,Chauncey Billups,42,0.512195,0.839924


## Most Surprising Seasons — Era Adjusted

The table above shows the 30 most surprising single seasons in the dataset — teams that won *significantly* more games than their stats predicted, adjusted so that every season is on a level playing field.

**Why era-adjust?** In the early seasons of the walk-forward model (1999–2003), the model has only 3–6 seasons of training data, so its predictions are less precise and residuals are naturally larger for everyone. Without adjustment, those early seasons would dominate the top-30 simply because the model was less calibrated, not because the teams were genuinely more surprising. The era adjustment (subtracting each season's mean residual from every team's residual that year) removes this bias — it only rewards teams that stood out *relative to other teams in the same season*, not relative to the model's overall accuracy.

**How to read it.** A high `residual_vs_era` score means the team won substantially more than their stats predicted *and* they did so in a season where the model was accurate for most other teams. That combination is the strongest signal that something beyond box-score stats was driving the result. The coach column identifies who was behind the wheel.

**What this is not.** A team appearing here does not mean it had the best season in NBA history — it means it had the most *unexplained* season. The 2016-17 Golden State Warriors (73-win season) would score low here because the model already predicted them to win in the mid-60s. A 50-win team predicted to win 35 is far more surprising to the model than a 67-win team predicted to win 64.

## How It All Fits Together

This notebook is a 30-year audit of the NBA that separates *talent* from *performance*. Every number answers one of two questions: how good were these players on paper, and how much did everything else turn that into actual wins?

**The pipeline in order:**

1. **Data collection** — 30 seasons of team stats (both offensive and opponent/defensive) pulled directly from the NBA API
2. **Walk-forward model** — predicts each season using only past data; residuals measure genuine out-of-sample surprise. The 17-feature model achieves an average R² of 0.862.
3. **Team charts** — show a franchise's full history of over- and underperformance alongside raw wins
4. **All-seasons model** — trains on all 30 years at once for fair cross-era coach comparisons
5. **League coaching table** — ranks every coach by their average residual vs the league mean
6. **Team coaching charts** — show all of a franchise's coaches on the same scale, vs both the team and the league
7. **Career chart** — zooms into a single coach's season-by-season story across all tenures
8. **Era-adjusted table** — highlights the single most surprising seasons in 30 years of NBA history

**The most important takeaway.** This model uses 17 statistical inputs — 9 offensive and 8 opponent/defensive — and deliberately leaves out star power, injuries, trades, schedule difficulty, and tanking. That is a feature, not a bug — the goal is not to build the most accurate win predictor, but to use the gap between prediction and reality to surface stories that pure stats cannot tell. The residual is a starting point for asking better questions, not a verdict.